# 01. Khảo sát dữ liệu Lending Club

Đếm dòng, đo tỷ lệ thiếu, và **đo mức rò rỉ bằng số** trước khi quyết định loại cột nào.
Kết luận của notebook này nằm ở chương 2 của báo cáo.

In [1]:
import sys; sys.path.insert(0, "..")
import numpy as np, pandas as pd
pd.set_option("display.width", 160)

In [2]:
from src.dataset import load_raw, LEAKING_COLUMNS, REDUNDANT_COLUMNS, TARGET
raw = load_raw("../data/raw/accepted_2007_to_2018Q4.csv.gz")
print(f"{len(raw):,} dòng x {raw.shape[1]} cột")
raw[TARGET].describe()

2,260,701 dòng x 54 cột


count    2.260668e+06
mean     1.309283e+01
std      4.832138e+00
min      5.310000e+00
25%      9.490000e+00
50%      1.262000e+01
75%      1.599000e+01
max      3.099000e+01
Name: int_rate, dtype: float64

## Tỷ lệ thiếu

In [3]:
miss = (raw.isna().mean() * 100).sort_values(ascending=False)
miss[miss > 1].to_frame("thiếu (%)").round(2)

,thiếu (%)
mths_since_recent_inq,13.07
emp_length,6.50
bc_util,3.37
percent_bc_gt_75,3.34
bc_open_to_buy,3.32
pct_tl_nvr_dlq,3.12
avg_cur_bal,3.11
mo_sin_old_rev_tl_op,3.11
num_rev_accts,3.11
num_il_tl,3.11


## Mức rò rỉ

`sub_grade` và `grade` là bảng tra lãi suất của Lending Club, nên chúng chính là biến
mục tiêu mang tên khác. Đo bằng $R^2$ của hồi quy `int_rate` theo riêng từng cột.

In [4]:
lc = pd.read_csv("../data/raw/accepted_2007_to_2018Q4.csv.gz",
                 usecols=["int_rate", "grade", "sub_grade", "issue_d"], low_memory=False).dropna(subset=["int_rate"])
for col in ("sub_grade", "grade"):
    resid = lc.int_rate - lc.groupby(col)["int_rate"].transform("mean")
    print(f"{col:<10} {lc[col].nunique():>3} mức   R^2 = {1 - resid.var()/lc.int_rate.var():.4f}")
y2016 = lc[lc.issue_d.str.endswith("2016", na=False)]
r = y2016.int_rate - y2016.groupby("sub_grade")["int_rate"].transform("mean")
print(f"sub_grade trong riêng 2016 ({len(y2016):,} dòng): R^2 = {1 - r.var()/y2016.int_rate.var():.4f}")

sub_grade   35 mức   R^2 = 0.9554
grade        7 mức   R^2 = 0.9088
sub_grade trong riêng 2016 (434,407 dòng): R^2 = 0.9834


In [5]:
print("Cột không bao giờ đọc vào:")
print(" rò rỉ    :", sorted(LEAKING_COLUMNS)[:8], "...")
print(" trùng lặp:", sorted(REDUNDANT_COLUMNS))

Cột không bao giờ đọc vào:
 rò rỉ    : ['collection_recovery_fee', 'debt_settlement_flag', 'grade', 'installment', 'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low', 'last_pymnt_amnt'] ...
 trùng lặp: ['fico_range_high', 'funded_amnt', 'funded_amnt_inv']
